# 14. Data quality and descriptive trends

## tl;dr

The processing layers pass their structural integrity checks, but buyer identifiers, duration, amount, and independent benchmark validation remain material limitations. Quarterly PELT results are descriptive break signals, not causal explanations or forecasts.

## Context & Methods

The unit is one awarded Grand Ouest digital procurement episode. The trend window starts at 2015Q2 because the raw extract begins in March 2015. PELT uses standardized quarterly counts and a log(n) penalty with sensitivity multipliers 0.5, 1, and 2.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'scripts').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data/processed/boamp'
with open(PROCESSED / 'data_quality_profile.json', encoding='utf-8') as f:
    quality = json.load(f)
with open(PROCESSED / 'trend_analysis_summary.json', encoding='utf-8') as f:
    trend_summary = json.load(f)
quarterly = pd.read_csv(PROCESSED / 'trend_quarterly.csv', parse_dates=['quarter_start'])
breakpoints = pd.read_csv(PROCESSED / 'trend_breakpoints.csv')
signals = pd.read_csv(PROCESSED / 'trend_signal_matrix.csv')


## Data

In [ ]:
pd.DataFrame([
    {'metric': 'standardized notices', 'value': quality['volume']['standardized_notices']},
    {'metric': 'reconstructed episodes', 'value': quality['volume']['reconstructed_episodes']},
    {'metric': 'study cohort episodes', 'value': quality['volume']['survival_cohort_rows']},
    {'metric': 'candidate pairs', 'value': quality['volume']['candidate_pairs']},
    {'metric': 'primary successor events', 'value': quality['volume']['accepted_primary_links']},
])

In [ ]:
pd.Series(quality['cohort_missingness'], name='missing_rate').sort_values(ascending=False).to_frame()

## Results

In [ ]:
display(signals)
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)
for ax, (segment, group) in zip(axes.flatten(), quarterly.groupby('segment', sort=False)):
    group = group.sort_values('quarter_start')
    ax.plot(group['quarter_start'], group['episode_count'], color='#356E9A')
    ax.set_title(segment)
    ax.grid(axis='y', alpha=0.25)
axes.flatten()[-1].axis('off')
fig.suptitle('Quarterly awarded digital procurement episodes')
plt.tight_layout()

In [ ]:
overall = quarterly.loc[quarterly['segment'].eq('Overall')].sort_values('quarter_start')
ax = overall.plot(x='quarter_start', y='duration_completeness', figsize=(10, 4), legend=False, color='#356E9A')
ax.set_title('Reliable duration completeness by quarter')
ax.set_ylabel('share')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.25)

## Takeaways

- The current project has enough episodes for descriptive survival analysis, but the event definition remains linkage-conditioned.
- Duration missingness changes sharply over time, so global duration imputation would create unsupported temporal structure.
- PELT breaks are candidates for documentary interpretation, not causal findings.
- Current benchmark metrics remain internal development evidence until independent specialist review is completed.